TITULO: 
Sistema automático de control de asistencia y gestión de ausencias con verificación de justificativos

RESUMEN: 
Detectar ausencias en tiempo real, solicitar la razón al empleado, recolectar y validar justificativos (p. ej., certificado médico) y notificar según reglas y plazos definidos.


PROBLEMA A RESOLVER
Implementar un sistema de control de las ausencias de los empleados con horarios definidos (turnos) a través de dos canales de contacto: WhatsApp/Email. Se los contactará por alguno de estos dos medios para que el empleado informe el motivo de su falta y envíe un justificatorio en caso de corresponder.

FUNCIÓN DE LA IA: 
Generar un contacto con el/la empleado/a y según la respuesta de este/a accionar de una manera determinada. Ejemplo:
 
1.	Detección de ausencia:

•	El sistema compara el horario esperado vs. marcación/ingreso (reloj, app, fichada). Si no hay check in antes de la hora de corte del turno, se dispara el caso de ausencia.

2.	Primer contacto automático:

•	Mensaje al empleado: “No registramos tu ingreso. ¿Cuál es el motivo?” con botones rápidos: [Enfermedad] [Turno cambiado] [Permiso aprobado] [Fuerza mayor] [Otro] + campo de texto opcional.

3.	Clasificación de motivo:

•	Si el motivo es Enfermedad: solicitar certificado médico y fecha de atención, con plazo de carga (p. ej., 24 h).

•	Si el motivo es Permiso aprobado: pedir número/ID de permiso o adjuntar constancia.

•	Otros: recolectar breve explicación o documento de soporte, si aplica.

4.	Subida de justificativo:

•	El empleado adjunta un documento/foto. El sistema extrae datos (OCR) y valida reglas básicas.

5.	Validación del justificativo (reglas):

•	Fecha: debe ser la fecha actual o dentro de X días del evento (configurable).

•	Legibilidad: nombre del empleado, fecha, profesional/centro, firma/sello o equivalente.

•	Originalidad/alteraciones: comprobación heurística (metadatos, detección de ediciones obvias). Cuando falle, derivar a revisión humana.

6.	Decisión y notificaciones:

•	Si es válido: marcar ausencia como justificada. Opción: notificar silenciosamente al empleador o enviar confirmación breve.

•	Si es no válido o falta cargarlo dentro del plazo: enviar aviso al empleado y al empleador; escalar si no hay respuesta tras X horas.




TECNICAS DE FAST PROMPTING a utilizar: 
1. Role Playing (Asignación de Persona): Se le da a la IA un rol específico (Ej: "experto en RRHH").
2. Instructional Prompting (Instrucciones Claras): Se da una orden directa y específica.
3. Zero-Shot Prompting: Se le pide a la IA que genere un mensaje sin darle ningún ejemplo previo de cómo debería ser.
4. Few-Shot Prompting (Para la clasificación de motivos): Para cuando un empleado elige "Otro motivo" y escribe una explicación. Se le da algunos ejemplos a la IA para guiarla y no dejar que interpete libremente. 
5.  Chain-of-Thought (CoT) Prompting (Para la validación de justificativos): Es una de las técnicas más potentes para tareas que requieren lógica y varios pasos de razonamiento. En lugar de pedir el resultado final directamente, le pides a la IA que "piense en voz alta" o "razone paso a paso". Ejemplo: Para analizar el texto extraído por el OCR de un certificado médico y tomar una decisión.
6. Structured Output Prompting (JSON/YAML): Es crucial para que tu sistema sea automático. En lugar de recibir un texto en lenguaje natural, se le exige a la IA que responda en un formato estructurado como JSON, que el código Python puede leer (parsear) directamente. Es el complemento perfecto para el Chain-of-Thought en el Paso 5.

In [ ]:
import os
import google.generativeai as genai
from dotenv import load_dotenv

load_dotenv()
    
api_key = os.getenv("GOOGLE_API_KEY")

def generar_mensaje_automatico_ia(nombre_empleado, empresa, horario, motivo_opcion):
    """
    Genera un mensaje único con IA y maneja los errores de la API correctamente.
    """
    try:
        motivos = {
            "1": "enfermedad o malestar de salud",
            "2": "permiso previamente aprobado",
            "3": "cambio de turno o horario",
            "4": "situación de fuerza mayor o emergencia",
            "5": "otro motivo no especificado"
        }
        motivo_desc = motivos.get(motivo_opcion, "otro motivo")
        
        prompt = f"""
        Eres un asistente de Recursos Humanos experto en comunicación laboral. 
        Genera UN ÚNICO mensaje de WhatsApp para {nombre_empleado} de la empresa {empresa}
        que no se presentó a trabajar a las {horario}.
        El empleado ha indicado que el motivo es: {motivo_desc}
        
        REQUISITOS DEL MENSAJE:
        - Totalmente único y natural, NO uses plantillas predefinidas.
        - Adaptado específicamente al motivo: {motivo_desc}.
        - Lenguaje empático pero profesional.
        - Incluye instrucciones claras de seguimiento según el motivo.
        - Usa 2-3 emojis relevantes naturalmente integrados.
        - Máximo 4 oraciones, estilo conversacional de WhatsApp.
        - Firma como "Equipo de RRHH - {empresa}".
        
        CREA UN MENSAJE COMPLETAMENTE ORIGINAL Y NATURAL.
        """
        model = genai.GenerativeModel("gemini-2.0-flash")
        
        # Generamos el contenido
        respuesta = model.generate_content(prompt)
        
        # --- ESTE ES EL CAMBIO CLAVE ---
        # Verificamos si la respuesta tiene texto antes de intentar acceder a él.
        # El acceso a .text puede fallar si la respuesta fue bloqueada.
        return respuesta.text.strip()

    except ValueError:
        # Esta excepción se lanza a menudo cuando el contenido es bloqueado por seguridad
        return (f"❌ ERROR: La respuesta de la IA fue bloqueada, probablemente por un filtro de seguridad.\n"
                f"   Feedback del bloqueo: {respuesta.prompt_feedback}")
    except Exception as e:
        # Captura cualquier otro error (ej. clave de API inválida, problema de red)
        return f"❌ ERROR GENERAL: Ocurrió un problema al contactar la API de Gemini.\n   Detalles: {e}"


def sistema_mensajes_automatico():
    """
    Sistema que procesa la respuesta del empleado. (Sin cambios aquí)
    """
    print("🤖 SISTEMA AUTOMÁTICO DE MENSAJES WHATSAPP")
    print("=" * 50)
    
    nombre_empleado = input("👤 Nombre del empleado: ").strip()
    horario = input("⏰ Horario esperado (ej: 08:00): ").strip()
    empresa = input("🏢 Empresa: ").strip()
    
    # ... (el resto de esta función sigue exactamente igual)
    
    print(f"\n📱 Enviando mensaje inicial a {nombre_empleado}...")
    mensaje_inicial = f"""
Hola {nombre_empleado}, 👋
No registramos tu ingreso a las {horario}. ¿Podrías indicarnos el motivo?
1️⃣ 🤒 Enfermedad
2️⃣ ✅ Permiso aprobado  
3️⃣ 🔄 Cambio de turno
4️⃣ 💥 Fuerza mayor
5️⃣ ❓ Otro motivo
Equipo de RRHH - {empresa}"""
    print("\n💬 MENSAJE INICIAL ENVIADO:")
    print("=" * 50)
    print(mensaje_inicial)
    
    while True:
        opcion_empleado = input("\n¿Qué opción elige el empleado? (1-5): ").strip()
        if opcion_empleado in ["1", "2", "3", "4", "5"]:
            break
        print("❌ Opción no válida. Elige 1-5")
    
    nombres_opciones = {
        "1": "Enfermedad 🤒", "2": "Permiso aprobado ✅", "3": "Cambio de turno 🔄", 
        "4": "Fuerza mayor 💥", "5": "Otro motivo ❓"
    }
    
    print(f"\n📋 Empleado seleccionó: {nombres_opciones[opcion_empleado]}")
    print("🔄 Generando respuesta automática con IA...")
    
    mensaje_respuesta = generar_mensaje_automatico_ia(
        nombre_empleado, empresa, horario, opcion_empleado
    )
    
    print(f"\n💬 MENSAJE AUTOMÁTICO GENERADO:")
    print("=" * 60)
    print(mensaje_respuesta) # AHORA SÍ MOSTRARÁ ALGO (EL MENSAJE O EL ERROR)
    print("=" * 60)
    
    print(f"\n🎉 Proceso completado!")
    input("\nPresiona Enter para salir...")

# Ejecutar el sistema
if __name__ == "__main__":
    sistema_mensajes_automatico()

🤖 SISTEMA AUTOMÁTICO DE MENSAJES WHATSAPP


c:\Users\buken\Documents\CODERHOUSE\IA_Generacion_de_Prompts_3.0\Entrega_Clase_IA\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm



📱 Enviando mensaje inicial a matias...

💬 MENSAJE INICIAL ENVIADO:

Hola matias, 👋
No registramos tu ingreso a las 9. ¿Podrías indicarnos el motivo?
1️⃣ 🤒 Enfermedad
2️⃣ ✅ Permiso aprobado  
3️⃣ 🔄 Cambio de turno
4️⃣ 💥 Fuerza mayor
5️⃣ ❓ Otro motivo
Equipo de RRHH - sec

📋 Empleado seleccionó: Permiso aprobado ✅
🔄 Generando respuesta automática con IA...

💬 MENSAJE AUTOMÁTICO GENERADO:
Hola Matías, 👋 Buen día! Vimos que no estás por la oficina hoy, esperamos que estés disfrutando tu día libre según el permiso aprobado. Si surge algún inconveniente o cambio con tu reincorporación, avísanos porfa! 🗓️ Equipo de RRHH - sec

🎉 Proceso completado!


CUANDO EL EMPLEADO ENVÍA EL COMPROBANTE MÉDICO, LA TECNOLOGÍA OCR EXTRAE EL TEXTO DE LA IMAGEN Y LUEGO "CHAIN-OF-THOUGHT" ANALIZA (MOSTRANDO EL MECANISMO DE EVALUACIÓN Y RAZONAMIENTO) SI EL JUSTIFICATIVO ES VÁLIDO O INVÁLIDO. 

In [5]:
import json

def validar_certificado_con_cot(texto_ocr: str, nombre_empleado_esperado: str, fecha_ausencia_esperada: str, documento_empleado: str, empresa: str) -> dict:
    """
    Analiza el texto de un certificado médico usando Chain-of-Thought
    y devuelve un análisis estructurado en JSON.
    """
    
    prompt = f"""
    Eres un auditor de RRHH extremadamente meticuloso para la empresa "{empresa}".
    Tu tarea es validar el siguiente texto, extraído vía OCR de un certificado médico,
    siguiendo un riguroso proceso de razonamiento paso a paso (Chain-of-Thought).

    DATOS A VALIDAR:
    - Nombre del empleado: "{nombre_empleado_esperado}"
    - Fecha de la ausencia: "{fecha_ausencia_esperada}"
    - Documento del empleado: "{documento_empleado}"

    TEXTO EXTRAÍDO DEL CERTIFICADO:
    ---
    {texto_ocr}
    ---

    INSTRUCCIONES:
    Primero, realiza tu razonamiento paso a paso de forma explícita. Sigue estos pasos:
    1.  **Verificación de Nombre:** Busca el nombre "{nombre_empleado_esperado}" en el texto. ¿Aparece? Anota las coincidencias o variaciones menores (ej. omisión de un segundo nombre).
    2.  **Extracción de Fecha:** Identifica y extrae la fecha de atención médica del texto.
    3.  **Comparación de Fecha:** Compara la fecha extraída con la fecha de ausencia esperada ("{fecha_ausencia_esperada}"). ¿Coinciden?
    4.  **Verificación de Documento:** ¿Se menciona el documento del empleado ("{documento_empleado}") en el texto? Anótalo como indicador de legitimidad.
    5.  **Verificación de Profesional/Centro:** ¿Se menciona a un profesional médico o un centro de salud? Anótalo como indicador de legitimidad.
    6.  **Conclusión Lógica:** Basado en los pasos anteriores, llega a una conclusión sobre la validez del documento.

    Finalmente, resume tu análisis completo en un ÚNICO objeto JSON. No escribas nada después del JSON.
    La estructura del JSON debe ser la siguiente:
    {{
      "razonamiento": "Aquí escribe tu análisis paso a paso...",
      "decision": "Elige una de estas tres opciones: 'VÁLIDO', 'INVÁLIDO', 'REQUIERE_REVISIÓN_MANUAL'",
      "nombre_encontrado": "El nombre que encontraste en el texto o null",
      "documento_encontrado": "El documento que encontraste en el texto o null",
      "fecha_encontrada": "La fecha que encontraste en el texto (formato YYYY-MM-DD) o null",
      "motivo_decision": "Una explicación breve y clara de por qué tomaste esa decisión."
    }}
    """
    
    try:
        model = genai.GenerativeModel("gemini-2.0-flash")
        respuesta = model.generate_content(prompt)
        
        # Extraer el bloque JSON de la respuesta de la IA
        respuesta_texto = respuesta.text
        json_start = respuesta_texto.find('{')
        json_end = respuesta_texto.rfind('}') + 1
        
        if json_start != -1 and json_end != -1:
            json_str = respuesta_texto[json_start:json_end]
            return json.loads(json_str)
        else:
            return {"error": "No se encontró un JSON válido en la respuesta de la IA."}

    except Exception as e:
        return {"error": f"Ocurrió un error al procesar con la IA: {e}"}

# --- EJEMPLO DE USO ---
if __name__ == "__main__":
    # --- CASO 1: Certificado VÁLIDO ---
    print("--- CASO 1: Certificado VÁLIDO ---")
    texto_ocr_valido = """
    Clínica Bienestar Total
    CERTIFICADO MÉDICO
    Se deja constancia que el paciente, Sr. Matias Bökenhans, DNI 30.492.910,
    fue atendido el día 17 de Septiembre de 2025 por un cuadro gripal.
    Se recomienda reposo por 48 horas.
    Dr. Roberto Fernández - Matrícula P. 78910
    """
    resultado_valido = validar_certificado_con_cot(
        texto_ocr=texto_ocr_valido,
        nombre_empleado_esperado="Matias Federico Bökenhans",
        fecha_ausencia_esperada="2025-09-17",
        documento_empleado= "30492910",
        empresa="SEC"
    )
    print("DECISIÓN FINAL:", resultado_valido.get("decision"))
    print("MOTIVO:", resultado_valido.get("motivo_decision"))
    print("\nRAZONAMIENTO DE LA IA (Chain-of-Thought):")
    print(resultado_valido.get("razonamiento"))
    print("="*60)

    # --- CASO 2: Certificado INVÁLIDO (Fecha incorrecta) ---
    print("\n--- CASO 2: Certificado INVÁLIDO (Fecha incorrecta) ---")
    texto_ocr_invalido = """
    Consultorio Dr. Gómez
    Por medio de la presente se certifica que Matias F. Bökenhans fue evaluado
    en consulta médica el día 10-09-2025.
    Firma: Dr. Gómez
    """
    resultado_invalido = validar_certificado_con_cot(
        texto_ocr=texto_ocr_invalido,
        nombre_empleado_esperado="Matias Federico Bökenhans",
        fecha_ausencia_esperada="17-09-2025", 
        documento_empleado="30492910",
        empresa="SEC"
    )
    print("DECISIÓN FINAL:", resultado_invalido.get("decision"))
    print("MOTIVO:", resultado_invalido.get("motivo_decision"))
    print("\nRAZONAMIENTO DE LA IA (Chain-of-Thought):")
    print(resultado_invalido.get("razonamiento"))
    print("="*60)

    # --- CASO 3: Requiere Revisión (Nombre con ligera variación) ---
    print("\n--- CASO 3: REQUIERE REVISIÓN (Nombre con variación) ---")
    texto_ocr_revision = """
    Centro Médico del Sur
    Certificamos que el paciente Matias F. Bokenhans fue atendido el 17/09/2025.
    Dr. L. Soto
    """
    resultado_revision = validar_certificado_con_cot(
        texto_ocr=texto_ocr_revision,
        nombre_empleado_esperado="Matias Federico Bökenhans", 
        fecha_ausencia_esperada="17-09-2025",
        documento_empleado="30492910",
        empresa="SEC"
    )
    print("DECISIÓN FINAL:", resultado_revision.get("decision"))
    print("MOTIVO:", resultado_revision.get("motivo_decision"))
    print("\nRAZONAMIENTO DE LA IA (Chain-of-Thought):")
    print(resultado_revision.get("razonamiento"))
    print("="*60)
    
    input("\nPresiona Enter para salir.")

--- CASO 1: Certificado VÁLIDO ---
DECISIÓN FINAL: REQUIERE_REVISIÓN_MANUAL
MOTIVO: El documento es similar a lo esperado pero difiere en el nombre (omisión del segundo nombre) y formato del DNI. Por lo tanto requiere revisión humana.

RAZONAMIENTO DE LA IA (Chain-of-Thought):
1. **Verificación de Nombre:** El nombre buscado es 'Matias Federico Bökenhans'. En el texto se encuentra 'Matias Bökenhans'. Hay una omisión del segundo nombre 'Federico'.
2. **Extracción de Fecha:** La fecha de atención médica extraída del texto es '17 de Septiembre de 2025'.
3. **Comparación de Fecha:** La fecha extraída '2025-09-17' coincide con la fecha de ausencia esperada '2025-09-17'.
4. **Verificación de Documento:** El documento del empleado buscado es '30492910'. En el texto se encuentra '30.492.910'. Si se remueven los puntos, coincide con el documento buscado.
5. **Verificación de Profesional/Centro:** El certificado menciona a 'Clínica Bienestar Total' y al 'Dr. Roberto Fernández - Matrícula P. 7891

SI LA PERSONA ELIGE "OTRO MOTIVO" LA IA DEBERÁ RECONOCER EL MOTIVO Y EMITIR UN TEXTO ACORDE. 

In [6]:
def clasificar_motivo_ia(texto_empleado: str) -> str:
    """
    Clasifica el motivo de ausencia usando la API de Gemini con Few-Shot Prompting.
    """
    prompt = f"""
    Eres un asistente de RRHH que clasifica los motivos de ausencia de los empleados en una de estas categorías: [Asunto Médico], [Trámite Personal], [Problema Familiar], [Incidente de Transporte], [Otro].

    Ejemplo 1:
    Texto del empleado: "Tuve que llevar a mi hijo al médico de urgencia."
    Categoría: [Problema Familiar]
    ---
    Ejemplo 2:
    Texto del empleado: "Se me pinchó una rueda en la autopista y no llegué a tiempo."
    Categoría: [Incidente de Transporte]
    ---
    Ejemplo 3:
    Texto del empleado: "Fui a renovar mi licencia de conducir."
    Categoría: [Trámite Personal]
    ---
    Ahora, clasifica el siguiente texto:
    Texto del empleado: "{texto_empleado}"
    Categoría:"""
    try:
        model = genai.GenerativeModel("gemini-2.0-flash")
        respuesta = model.generate_content(prompt)
        # Limpiamos la respuesta para obtener solo el texto de la categoría
        categoria = respuesta.text.strip().replace("[", "").replace("]", "")
        return categoria
    except Exception as e:
        print(f"⚠️ Error al clasificar: {e}")
        return "Otro" # Devolvemos 'Otro' como fallback en caso de error

def generar_mensaje_post_clasificacion(nombre_empleado, empresa, categoria_clasificada, texto_original_empleado):
    """
    Genera un mensaje de seguimiento basado en la categoría clasificada por la IA.
    """
    prompt = f"""
    Eres un asistente de RRHH de la empresa "{empresa}". Tu tarea es escribir una respuesta de WhatsApp para {nombre_empleado}.
    El empleado seleccionó 'Otro motivo' y explicó lo siguiente: "{texto_original_empleado}".
    Nuestro sistema ha clasificado su motivo como: "{categoria_clasificada}".

    Basado en esa clasificación, escribe un mensaje empático y claro:
    - Si es Asunto Médico o Problema Familiar, muestra preocupación y menciona que revisarás el caso para ver si se necesita un justificativo.
    - Si es Trámite Personal o Incidente de Transporte, sé comprensivo y pregunta si necesita algún comprobante para adjuntar.
    - Si la categoría es "Otro", simplemente acusa recibo y menciona que el equipo de RRHH lo revisará.

    El mensaje debe ser breve, profesional y sonar natural. Usa 1 o 2 emojis.
    """
    try:
        model = genai.GenerativeModel("gemini-2.0-flash")
        respuesta = model.generate_content(prompt)
        return respuesta.text.strip()
    except Exception as e:
        return f"❌ Error al generar el mensaje de respuesta: {e}"


def simular_flujo_otro_motivo():
    """
    Función principal que ejecuta el flujo completo para la opción 5.
    """
    print("🤖 Simulador de Respuesta para 'Otro Motivo' (Opción 5)")
    print("=" * 60)
    
    # 1. Recolectar datos
    nombre_empleado = input("👤 Nombre del empleado: ").strip()
    empresa = input("🏢 Empresa: ").strip()
    texto_libre_empleado = input("📝 Escribe la explicación que dio el empleado: ").strip()

    # 2. Clasificar el motivo
    print("\n🔍 Analizando y clasificando el motivo con IA...")
    categoria = clasificar_motivo_ia(texto_libre_empleado)
    print(f"✅ Motivo clasificado como: '{categoria}'")

    # 3. Generar la respuesta final
    print("\n🔄 Generando respuesta personalizada...")
    mensaje_final = generar_mensaje_post_clasificacion(
        nombre_empleado, empresa, categoria, texto_libre_empleado
    )
    
    # 4. Mostrar resultado
    print("\n" + "=" * 60)
    print("💬 MENSAJE FINAL GENERADO PARA ENVIAR:")
    print("-" * 35)
    print(mensaje_final)
    print("=" * 60)
    
    input("\nPresiona Enter para salir.")

# Ejecutar el simulador
if __name__ == "__main__":
    simular_flujo_otro_motivo()

🤖 Simulador de Respuesta para 'Otro Motivo' (Opción 5)

🔍 Analizando y clasificando el motivo con IA...
✅ Motivo clasificado como: 'Incidente de Transporte'

🔄 Generando respuesta personalizada...

💬 MENSAJE FINAL GENERADO PARA ENVIAR:
-----------------------------------
¡Hola Matías! 👋 Entendemos que estás atascado en la General Paz. Comprendemos la situación. ¿Necesitas algún comprobante para adjuntar como respaldo?


In [ ]:
import os
import vertexai
from vertexai.preview.vision_models import ImageGenerationModel
from PIL import Image, ImageDraw, ImageFont

# ================= CONFIGURACIÓN =================
PROJECT_ID = os.getenv("GOOGLE_PROJECT_ID")
LOCATION = os.getenv("GOOGLE_LOCATION ")

# ================= FUNCIONES PRINCIPALES =================

def verificar_autenticacion():
    """Verifica que la autenticación con Google Cloud funciona."""
    try:
        from google.auth import default
        credentials, project_id = default()
        
        # Detectar el tipo de credenciales
        if hasattr(credentials, 'service_account_email'):
            # Es una cuenta de servicio
            auth_type = "Service Account"
            email = credentials.service_account_email
        else:
            # Es una credencial de usuario
            auth_type = "User Credentials"
            email = credentials.quota_project_id or "Usuario autenticado"
        
        print("✅ Autenticación exitosa.")
        print(f"   - Tipo: {auth_type}")
        print(f"   - Identidad: {email}")
        print(f"   - Proyecto: {project_id or PROJECT_ID}")
        
        return True
    except Exception as e:
        print(f"❌ Error de autenticación: {e}")
        print("\nSolución:")
        print("1. Para cuenta de servicio:")
        print("   export GOOGLE_APPLICATION_CREDENTIALS='ruta/credenciales.json'")
        print("2. Para credenciales de usuario:")
        print("   gcloud auth application-default login")
        return False

def inicializar_vertex_ai():
    """Inicializa la conexión con Vertex AI y maneja errores comunes."""
    try:
        print("Conectando con Google Vertex AI...")
        vertexai.init(project=PROJECT_ID, location=LOCATION)
        print("¡Conexión exitosa!")
        return True
    except Exception as e:
        print("\n" + "="*50)
        print("¡ERROR DE CONFIGURACIÓN!")
        print(f"No se pudo inicializar Vertex AI: {e}")
        print("\nPosibles soluciones:")
        print("1. Verifica que el proyecto existe y está activo")
        print("2. Asegura que las APIs de Vertex AI están habilitadas")
        print("3. Verifica los permisos de la cuenta")
        print("="*50 + "\n")
        return False

def generar_imagen_base(motivo_ausencia: str) -> str | None:
    """Genera únicamente la imagen de fondo con IA, SIN texto."""
    print(f"\nGenerando imagen de fondo para motivo: '{motivo_ausencia}'...")
    
    # Determinar el contexto visual basado en el motivo
    motivo_lower = motivo_ausencia.lower()
    contexto_visual = ""
    
    if "enfermedad" in motivo_lower or "médico" in motivo_lower or "medico" in motivo_lower:
        contexto_visual = "Una escena tranquila y rejuvenecedora, con una planta de oficina creciendo saludablemente y luz solar suave. Colores pastel suaves."
    elif "vacaciones" in motivo_lower or "viaje" in motivo_lower:
        contexto_visual = "Una escena que transmite energía de un nuevo comienzo, como un escritorio ordenado con una taza de café humeante y un cuaderno. Colores vibrantes."
    elif "permiso" in motivo_lower or "personal" in motivo_lower:
        contexto_visual = "Una escena de apoyo y equipo, como piezas de un rompecabezas uniéndose o manos estilizadas colaborando. Ambiente profesional y positivo."
    else:
        contexto_visual = "Un concepto abstracto de bienvenida, con formas geométricas amigables y un fondo limpio y corporativo."

    prompt_final = (
        f"Una ilustración digital para una tarjeta de bienvenida. {contexto_visual} "
        "Estilo alegre, profesional y minimalista, con espacio vacío en el centro. "
        "Formato horizontal, relación de aspecto 16:9."
    )
    
    negative_prompt = "texto, letras, escritura, personas, rostros, firmas, marcas de agua, borroso, feo, deforme"
    
    print(f"Prompt enviado a la IA: \"{prompt_final}\"")

    try:
        # Cargar el modelo de generación de imágenes
        print("Cargando modelo imagegeneration@006...")
        model = ImageGenerationModel.from_pretrained("imagegeneration@006")
        
        # Generar la imagen - SIN seed porque no es compatible con watermark
        print("Solicitando generación de imagen...")
        response = model.generate_images(
            prompt=prompt_final,
            negative_prompt=negative_prompt,
            number_of_images=1,
            # guidance_scale=7  # También comentado por si causa problemas
        )
        
        # Verificar si se generó alguna imagen
        if not response.images:
            print("\n" + "="*50)
            print("¡ATENCIÓN! La IA no devolvió ninguna imagen.")
            print("Posibles causas:")
            print("  1. El prompt activó filtros de seguridad de Google")
            print("  2. Problema temporal avec la API")
            print("  3. Cuota excedida o permisos insuficientes")
            print("="*50 + "\n")
            return None

        # Guardar la imagen generada
        imagen_generada = response.images[0]
        ruta_imagen_base = "temp_background_image.png"
        
        print("Guardando imagen generada...")
        imagen_generada.save(location=ruta_imagen_base)
        
        # Verificar que el archivo se creó correctamente
        if os.path.exists(ruta_imagen_base) and os.path.getsize(ruta_imagen_base) > 0:
            print(f"✅ Imagen base guardada: {ruta_imagen_base} ({os.path.getsize(ruta_imagen_base)} bytes)")
            return ruta_imagen_base
        else:
            print("❌ Error: El archivo de imagen no se creó correctamente")
            return None

    except Exception as e:
        print(f"🔥 Error durante la generación de imagen: {e}")
        import traceback
        traceback.print_exc()
        return None

def agregar_texto_a_imagen(ruta_imagen_base: str, mensaje: str, nombre_empleado: str) -> str:
    """Agrega texto personalizado a la imagen generada."""
    print("Agregando texto a la imagen...")
    
    try:
        # Intentar cargar una fuente bonita con tamaño más grande
        try:
            # Tamaño de fuente más grande (80 en lugar de 60)
            fuente = ImageFont.truetype("Montserrat-Bold.ttf", size=60)
        except IOError:
            try:
                # Intentar con Arial si Montserrat no está disponible
                fuente = ImageFont.truetype("arialbd.ttf", size=60)
            except IOError:
                try:
                    # Intentar con otra fuente común
                    fuente = ImageFont.truetype("DejaVuSans-Bold.ttf", size=60)
                except IOError:
                    print("ℹ️  No se encontraron fuentes personalizadas. Usando fuente por defecto.")
                    fuente = ImageFont.load_default()
        
        # Abrir la imagen base
        imagen = Image.open(ruta_imagen_base).convert("RGBA")
        draw = ImageDraw.Draw(imagen)
        
        # Calcular dimensiones de la imagen
        ancho_img, alto_img = imagen.size
        
        # Calcular tamaño del texto
        bbox = draw.textbbox((0, 0), mensaje, font=fuente, align="center")
        ancho_texto = bbox[2] - bbox[0]
        alto_texto = bbox[3] - bbox[1]
        
        # Posicionar el texto en la PARTE SUPERIOR (20% desde arriba)
        posicion_x = (ancho_img - ancho_texto) / 2
        posicion_y = alto_img * 0.10  # 20% desde la parte superior
        
        # Ajustar el tamaño si el texto es muy ancho para la imagen
        if ancho_texto > ancho_img * 0.9:  # Si ocupa más del 90% del ancho
            print("⚠️  El texto es muy ancho, reduciendo tamaño de fuente...")
            # Reducir tamaño de fuente progresivamente hasta que quepa
            tamaño_fuente = 60
            while ancho_texto > ancho_img * 0.9 and tamaño_fuente > 40:
                tamaño_fuente -= 5
                try:
                    fuente = ImageFont.truetype("Montserrat-Bold.ttf", size=tamaño_fuente)
                except:
                    try:
                        fuente = ImageFont.truetype("arialbd.ttf", size=tamaño_fuente)
                    except:
                        pass
                bbox = draw.textbbox((0, 0), mensaje, font=fuente, align="center")
                ancho_texto = bbox[2] - bbox[0]
                alto_texto = bbox[3] - bbox[1]
                posicion_x = (ancho_img - ancho_texto) / 2
        
        # Crear un fondo semitransparente para mejor legibilidad
        padding = 20
        fondo_rect = [
            posicion_x - padding,
            posicion_y - padding,
            posicion_x + ancho_texto + padding,
            posicion_y + alto_texto + padding
        ]
                
        # Agregar sombra al texto (más pronunciada)
        draw.text((posicion_x + 4, posicion_y + 4), mensaje, font=fuente, 
                 fill=(0, 0, 0, 200), align="center")  # Sombra más oscura
        
        # Texto principal - usar color que contraste bien
        draw.text((posicion_x, posicion_y), mensaje, font=fuente, 
                 fill=(255, 255, 255, 255), align="center")  # Blanco sólido
        
        # Guardar imagen final
        ruta_final = f"bienvenida_{nombre_empleado.lower().replace(' ', '_')}.png"
        imagen.save(ruta_final, "PNG")
        
        # Limpiar archivo temporal
        if os.path.exists(ruta_imagen_base):
            os.remove(ruta_imagen_base)
        
        print(f"✅ Imagen final guardada como: {ruta_final}")
        print(f"   - Tamaño de texto: {ancho_texto}x{alto_texto}px")
        print(f"   - Posición: ({posicion_x:.0f}, {posicion_y:.0f})")
        
        return ruta_final
        
    except Exception as e:
        print(f"❌ Error al agregar texto: {e}")
        import traceback
        traceback.print_exc()
        return ""

def main():
    """Función principal que ejecuta el bucle interactivo."""
    
    print("\n" + "="*60)
    print("GENERADOR DE IMÁGENES DE BIENVENIDA - VERTEX AI")
    print("="*60)
    
    # Verificar autenticación primero
    if not verificar_autenticacion():
        return
    
    # Inicializar Vertex AI
    if not inicializar_vertex_ai():
        return

    while True:
        try:
            # Obtener datos del usuario
            print("\n" + "-"*40)
            nombre = input("Introduce el nombre del empleado: ").strip()
            if not nombre:
                print("El nombre no puede estar vacío.")
                continue
            
            print("\nSugerencias: Enfermedad, Vacaciones, Permiso, Personal")
            motivo = input(f"Introduce el motivo de la ausencia de {nombre}: ").strip()
            if not motivo:
                print("El motivo no puede estar vacío.")
                continue
            
            # Generar imagen base
            ruta_base = generar_imagen_base(motivo)
            
            if ruta_base:
                # Personalizar mensaje según el motivo
                motivo_lower = motivo.lower()
                
                if "enfermedad" in motivo_lower or "medico" in motivo_lower or "médico" in motivo_lower:
                    mensaje_texto = f"¡Nos alegra que estés mejor,\n{nombre}!"
                elif "vacaciones" in motivo_lower or "viaje" in motivo_lower:
                    mensaje_texto = f"¡Bienvenido de vuelta,\n{nombre}!"
                elif "permiso" in motivo_lower or "personal" in motivo_lower:
                    mensaje_texto = f"¡Qué bueno tenerte de vuelta,\n{nombre}!"
                else:
                    mensaje_texto = f"¡Bienvenido de vuelta,\n{nombre}!"
                
                # Agregar texto a la imagen
                ruta_final = agregar_texto_a_imagen(ruta_base, mensaje_texto, nombre)
                
                if ruta_final and os.path.exists(ruta_final):
                    print(f"\n🎉 ¡Imagen creada exitosamente!")
                    print(f"📁 Archivo: {ruta_final}")
                    print("¡Puedes abrirla para ver el resultado!")
                else:
                    print("❌ No se pudo crear la imagen final.")
            else:
                print("❌ No se pudo generar la imagen base. Intenta con otro motivo.")
            
            # Preguntar si desea continuar
            continuar = input("\n¿Deseas generar otra imagen? (s/n): ").lower().strip()
            if continuar not in ['s', 'si', 'sí', 'yes']:
                break
                
        except KeyboardInterrupt:
            print("\n\n👋 Programa interrumpido por el usuario.")
            break
        except Exception as e:
            print(f"❌ Error inesperado: {e}")
            continue
            
    print("\n¡Gracias por usar el generador de imágenes! 👋")

# ================= EJECUCIÓN =================
if __name__ == "__main__":
    main()


GENERADOR DE IMÁGENES DE BIENVENIDA - VERTEX AI
✅ Autenticación exitosa.
   - Tipo: User Credentials
   - Identidad: creacionimagenes-471902
   - Proyecto: creacionimagenes-471902
Conectando con Google Vertex AI...
¡Conexión exitosa!

----------------------------------------

Sugerencias: Enfermedad, Vacaciones, Permiso, Personal

Generando imagen de fondo para motivo: 'enfermedad'...
Prompt enviado a la IA: "Una ilustración digital para una tarjeta de bienvenida. Una escena tranquila y rejuvenecedora, con una planta de oficina creciendo saludablemente y luz solar suave. Colores pastel suaves. Estilo alegre, profesional y minimalista, con espacio vacío en el centro. Formato horizontal, relación de aspecto 16:9."
Cargando modelo imagegeneration@006...
Solicitando generación de imagen...
Guardando imagen generada...
✅ Imagen base guardada: temp_background_image.png (3097813 bytes)
Agregando texto a la imagen...
✅ Imagen final guardada como: bienvenida_lorena.png
   - Tamaño de texto: 81

RESULTADOS: 
Se logra el objetivo propuesto. Al enviarle al empleado un mensaje por whatsapp se acortan los tiempos y se acelera el proceso del envío de documentación. 
Como se puede apreciar, el código logra su cometido generando el mensaje que se busca. En caso de enfermedad, genera un mensaje personalizado pidiéndole con delicadeza a la persona que envíe el comprobante. Cuando lo envía la IA analiza el comprobante y determina si es válida o no. En el caso que no pueda, aconseja que sea evaluado por una persona. 
Si el/la empleado/a escoge "OTRO MOTIVO", la IA evaluará el texto ingresado y determinará la naturaleza de ese motivo para luego evaluar los pasos a seguir. 

CONCLUSIONES:
Este proyecto busca resolver un problema real y muy común en las empresas. La automatización de este proceso ahorra una cantidad enorme de tiempo y reduce errores. Como se indica en "Resultados", se logra el objetivo y se demuestra lo práctica que es la IA bien utilizada y aplicada a tareas específicas acortando y acelerando los tiempos de muchas situaciones que antes no se resolvían hasta que la persona se reincorporaba al trabajo.



